In [6]:
import os
import pandas as pd
from google.colab import drive
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Directory Paths
base_dir = '/content/drive/MyDrive/Fall Semester 2026/AI Resource Capstone - 2277/Capstone Shared Folder/Datasets/Cleaned Data'
master_file = os.path.join(base_dir, 'master_living_costs_3counties.parquet')
output_file = os.path.join(base_dir, 'features_scaled_for_clustering.parquet')

# 3. Load Master Feature Store
print(f"Loading Master Feature Store from:\n{master_file}")
df = pd.read_parquet(master_file)
df['zip_code'] = df['zip_code'].astype(str).str.zfill(5)

# Keep primary keys and county names separate as metadata
zip_metadata = df[['zip_code', 'county_name']].copy()

# 4. Define Clustering Feature Sets (Excluding constants and text strings)
numeric_cols = [
    'nfip_total_claims',
    'nfip_avg_building_claim',
    'nfip_avg_contents_claim',
    'pct_area_100yr_flood',
    'pct_area_coastal_surge',
    'elec_base_monthly_fee',
    'elec_delivery_rate_per_kwh',
    'gas_distribution_rate_per_ccf',
    'water_base_monthly_fee',
    'water_rate_per_1k_gal',
    'adj_monthly_food_cost_moderate'
]

categorical_cols = [
    'est_flood_risk_tier',
    'elec_provider',
    'water_service_type'
]

passthrough_binary_cols = [
    'water_mud_flag',
    'twia_mandatory'
]

# 5. Build ColumnTransformer Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('pass', 'passthrough', passthrough_binary_cols)
    ],
    remainder='drop' # Explicitly drops unselected columns (gas_provider string, USDA constants, res_ratio)
)

# 6. Fit and Transform
X_scaled = preprocessor.fit_transform(df)
clean_feature_names = preprocessor.get_feature_names_out()

# 7. Format Output DataFrame
df_clean = pd.DataFrame(X_scaled, columns=clean_feature_names)
df_clean.insert(0, 'zip_code', zip_metadata['zip_code'].values)

# 8. Export to Parquet
df_clean.to_parquet(output_file, index=False)

print("\n--- Pipeline Verification ---")
print(f"Records Processed: {len(df_clean)} ZIP codes")
print(f"Clustering Dimensions: {df_clean.shape[1] - 1} numeric features")
print(f"Non-numeric columns remaining: {[c for c in df_clean.columns if c != 'zip_code' and df_clean[c].dtype == 'object']}")
print(f"Saved clean scaled feature store to:\n{output_file}")
df_clean.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading Master Feature Store from:
/content/drive/MyDrive/Fall Semester 2026/AI Resource Capstone - 2277/Capstone Shared Folder/Datasets/Cleaned Data/master_living_costs_3counties.parquet

--- Pipeline Verification ---
Records Processed: 279 ZIP codes
Clustering Dimensions: 20 numeric features
Non-numeric columns remaining: []
Saved clean scaled feature store to:
/content/drive/MyDrive/Fall Semester 2026/AI Resource Capstone - 2277/Capstone Shared Folder/Datasets/Cleaned Data/features_scaled_for_clustering.parquet


,zip_code,num__nfip_total_claims,num__nfip_avg_building_claim,num__nfip_avg_contents_claim,num__pct_area_100yr_flood,num__pct_area_coastal_surge,num__elec_base_monthly_fee,num__elec_delivery_rate_per_kwh,num__gas_distribution_rate_per_ccf,num__water_base_monthly_fee,...,num__adj_monthly_food_cost_moderate,cat__est_flood_risk_tier_AE,cat__est_flood_risk_tier_VE,cat__est_flood_risk_tier_X,cat__elec_provider_CenterPoint,cat__elec_provider_TNMP,cat__water_service_type_Municipal_CityOfHouston,cat__water_service_type_Suburban_MUD,pass__water_mud_flag,pass__twia_mandatory
0,77001,-0.584585,-0.519506,-0.355936,-0.350812,-0.221071,-0.306786,-0.306786,-0.645169,-0.645169,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
1,77002,-0.416165,1.297771,0.003322,-0.350812,-0.221071,-0.306786,-0.306786,-0.645169,-0.645169,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
2,77003,-0.565398,-0.711730,-0.253305,-0.350812,-0.221071,-0.306786,-0.306786,-0.645169,-0.645169,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
3,77004,-0.170997,-0.126061,-0.310168,-0.350812,-0.221071,-0.306786,-0.306786,-0.645169,-0.645169,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
4,77005,-0.021765,-0.262915,-0.093058,-0.350812,-0.221071,-0.306786,-0.306786,-0.645169,-0.645169,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
